In [0]:
%pip install -q google-genai
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("gemini_key", "")

In [0]:
from pyspark.sql import functions as F

DB, TOP_N = "workspace.default", 5

def pick(df, options):
    """Return the first column name from options that exists in df."""
    return next((c for c in options if c in df.columns), None)

risk   = spark.read.table(f"{DB}.gold_risk")
events = spark.read.table(f"{DB}.bronze_events")

EMP   = pick(risk,   ["employee_id", "user_id", "emp_id"])
SCORE = pick(risk,   ["risk_score", "risk", "anomaly_score", "score"])
RDATE = pick(risk,   ["date", "event_date", "activity_date", "day"])
TS    = pick(events, ["timestamp", "event_time", "event_timestamp", "ts", "datetime", "date"])
print(f"Columns used → employee: {EMP}, score: {SCORE}, risk date: {RDATE}, event time: {TS}")

top = (risk.groupBy(EMP)
           .agg(F.max(SCORE).alias("max_risk"))
           .orderBy(F.desc("max_risk"))
           .limit(TOP_N)
           .toPandas())
display(top)

Columns used → employee: employee_id, score: risk_score, risk date: date, event time: timestamp


employee_id,max_risk
137,0.8667
124,0.7676
79,0.7674
25,0.728
131,0.7272


In [0]:
def get_evidence(emp_id, max_events=60):
    # Their 3 highest-risk days (if the risk table has dates)
    emp_risk = risk.filter(F.col(EMP) == emp_id).orderBy(F.desc(SCORE))
    risk_rows = emp_risk.limit(3).toPandas().astype(str)

    emp_events = events.filter(F.col(EMP) == emp_id)
    if RDATE and TS:
        bad_days = [r for r in risk_rows[RDATE]]
        emp_events = emp_events.filter(F.to_date(F.col(TS)).cast("string").isin(bad_days))
    if TS:
        emp_events = emp_events.orderBy(TS)

    ev = emp_events.limit(max_events).toPandas().astype(str)
    ev.insert(0, "event_ref", [f"E{i+1}" for i in range(len(ev))])   # IDs Gemini must cite

    # Their normal behavior, for comparison
    normal = (risk.filter(F.col(EMP) == emp_id)
                  .agg(F.avg(SCORE).alias("avg_risk"), F.count("*").alias("days_observed"))
                  .toPandas().astype(str))
    return risk_rows, ev, normal

risk_rows, ev, normal = get_evidence(top.iloc[0][EMP])
display(ev.head(10))   # preview the evidence for the riskiest employee

event_ref,employee_id,department,timestamp,activity,file_size_bytes,email_external,usb_device_id,is_confidential
E1,137,Engineering,2023-10-06 00:00:00,email_external,nan,external,None,False
E2,137,Engineering,2023-10-06 01:00:00,file_open,2611196.0,None,None,False
E3,137,Engineering,2023-10-06 02:00:00,email_external,nan,internal,None,False
E4,137,Engineering,2023-10-06 03:00:00,file_open,5000796.0,None,None,False
E5,137,Engineering,2023-10-06 04:00:00,file_open,3908477.0,None,None,False
E6,137,Engineering,2023-10-06 05:00:00,logon,nan,None,None,False
E7,137,Engineering,2023-10-06 05:00:00,email_external,nan,external,None,False
E8,137,Engineering,2023-10-06 06:00:00,usb_connect,nan,None,USB-8314,False
E9,137,Engineering,2023-10-06 06:00:00,file_open,1906511.0,None,None,False
E10,137,Engineering,2023-10-06 07:00:00,logon,nan,None,None,False


In [0]:
import json, time
from google.genai import types

PROMPT = """You are a cautious insider-threat analyst at a company's security team.
All data is synthetic. Analyze the evidence below for ONE employee.

Rules:
- Only use facts from the evidence. Cite event_ref IDs (e.g. E3, E7) for every claim.
- Do NOT claim the person is guilty. Describe patterns that warrant review.
- Always give at least one plausible innocent explanation.

Return JSON with exactly these keys:
  "severity": "Low" | "Medium" | "High" | "Critical",
  "summary": 2-3 sentence plain-English summary,
  "what_happened": list of short bullet strings (in time order, with event_refs),
  "why_unusual": list of short bullet strings comparing to their normal behavior,
  "innocent_explanations": list of strings,
  "next_steps": list of recommended investigation steps,
  "cited_events": list of event_ref IDs you used

EMPLOYEE: {emp}
MAX RISK SCORE: {score}

NORMAL BEHAVIOR BASELINE:
{normal}

HIGHEST-RISK DAYS (from the detection model):
{risk_rows}

EVENTS ON THOSE DAYS:
{events}
"""

def investigate(emp_id, score):
    risk_rows, ev, normal = get_evidence(emp_id)
    prompt = PROMPT.format(
        emp=emp_id, score=round(float(score), 3),
        normal=normal.to_csv(index=False),
        risk_rows=risk_rows.to_csv(index=False),
        events=ev.to_csv(index=False),
    )

    models = [GEMINI_MODEL, "gemini-3.8-flash"]
    report = None
    for attempt in range(6):
        model = models[attempt % len(models)]          # rotate models if one is busy
        try:
            resp = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json", temperature=0.2),
            )
            report = json.loads(resp.text)
            report["model_used"] = model
            break
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"  {model} busy, retrying in {wait}s ({str(e)[:60]})")
            time.sleep(wait)

    if report is None:
        return None

    # Grounding check: do the cited events actually exist?
    real_refs = set(ev["event_ref"])
    cited = report.get("cited_events", [])
    grounded = sum(1 for c in cited if c in real_refs)
    report["grounding"] = f"{grounded}/{len(cited)} cited events verified"
    return report

In [0]:
import json, time, datetime
from google import genai
from google.genai import types

GEMINI_MODEL = "gemini-3.8-flash"
client = genai.Client(api_key=dbutils.secrets.get("dataghost", "gemini_key"))

rows = []
for _, r in top.iterrows():
    emp_id, score = r[EMP], r["max_risk"]
    print(f"🔎 Investigating {emp_id} ...")
    rep = investigate(emp_id, score)
    if rep is None:
        print(f"  ⚠️ skipped {emp_id}")
        continue
    rows.append({
        "employee_id":           str(emp_id),
        "risk_score":            float(score),
        "severity":              rep.get("severity", ""),
        "summary":               rep.get("summary", ""),
        "what_happened":         "\n".join(rep.get("what_happened", [])),
        "why_unusual":           "\n".join(rep.get("why_unusual", [])),
        "innocent_explanations": "\n".join(rep.get("innocent_explanations", [])),
        "next_steps":            "\n".join(rep.get("next_steps", [])),
        "grounding":             rep["grounding"],
        "model":                 GEMINI_MODEL,
        "generated_at":          datetime.datetime.utcnow().isoformat(),
    })
    time.sleep(10)   # stay under free-tier rate limits

reports = spark.createDataFrame(rows)
reports.write.mode("overwrite").option("overwriteSchema", "true") \
       .saveAsTable(f"{DB}.incident_reports")
print(f"✅ Saved {len(rows)} reports to {DB}.incident_reports")
display(reports)

🔎 Investigating 137.0 ...
  gemini-3.8-flash busy, retrying in 5s (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This mo)


/home/spark-e54230bb-3f38-4cc6-9767-05/.ipykernel/67/command-7595439636848303-1739374596:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at":          datetime.datetime.utcnow().isoformat(),


🔎 Investigating 124.0 ...


/home/spark-e54230bb-3f38-4cc6-9767-05/.ipykernel/67/command-7595439636848303-1739374596:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at":          datetime.datetime.utcnow().isoformat(),


🔎 Investigating 79.0 ...


/home/spark-e54230bb-3f38-4cc6-9767-05/.ipykernel/67/command-7595439636848303-1739374596:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at":          datetime.datetime.utcnow().isoformat(),


... (2 outputs left out)

🔎 Investigating 131.0 ...


/home/spark-e54230bb-3f38-4cc6-9767-05/.ipykernel/67/command-7595439636848303-1739374596:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at":          datetime.datetime.utcnow().isoformat(),


✅ Saved 5 reports to workspace.default.incident_reports


employee_id,generated_at,grounding,innocent_explanations,model,next_steps,risk_score,severity,summary,what_happened,why_unusual
137.0,2026-11-08T00:01:41.717206,39/39 cited events verified,"The employee may have been responding to an urgent engineering issue, system outage, or tight deployment deadline requiring extended overnight work. Connecting multiple USB devices could be a legitimate engineering task involving hardware debugging, flashing firmware, or moving development images across lab devices. External emails may represent standard project coordination with authorized external vendors, contractors, or clients.",gemini-3.8-flash,"Review the specific confidential files opened in E47 and E49 to assess sensitivity and verify business justification. Inspect the recipient domains and attachments associated with external emails sent on 2023-10-06 and 2023-10-07 (E1, E7, E12, E13, E20, E22, E24, E25, E28, E42, E50, E56, E58, E60). Cross-reference the identifiers of the connected USB drives (E8, E14, E17, E21, E32, E44, E45, E51) with corporate asset management to confirm if they are company-approved devices. Speak with the employee's engineering manager to determine whether project deadlines, on-call support, or shift handovers explain the continuous after-hours activity.",0.8667,High,"Employee 137 exhibited a significant spike in risk score (0.8667) on 2023-10-07, driven by high volumes of after-hours work and access to confidential files. Activity logs indicate continuous round-the-clock operations spanning October 6 and October 7, accompanied by connections to multiple distinct USB devices and regular external emailing. While these patterns warrant review for potential policy violations or data exposure, they may also stem from intensive engineering testing or urgent deployment demands.","During the early morning hours of 2023-10-06, the employee accessed files (E2, E4, E5), sent an external email (E1, E7), and connected USB device USB-8314 (E8). Throughout the business day on 2023-10-06, the employee sent several external emails (E12, E13, E20) and connected additional USB devices USB-5789 (E14), USB-6773 (E17), and USB-5809 (E21). Late on 2023-10-06 and into the early morning of 2023-10-07, external emails continued (E22, E24, E25, E28), alongside a confidential USB connection for USB-1331 (E32) and multiple file opens (E27, E29, E30, E34, E37, E39, E40, E41, E43). On 2023-10-07, the employee connected more USB drives including USB-1799 (E44), USB-1814 (E45), and USB-3470 (E51). The employee accessed confidential files during the day on 2023-10-07 at 10:00 (E47) and 12:00 (E49), while continuing to send external emails (E42, E50, E56, E58, E60) and opening non-confidential files (E55, E57, E59).","The daily risk score reached 0.8667 on 2023-10-07, which is substantially above the 30-day baseline average risk of 0.24026. After-hours activity was unusually high on 2023-10-07 with 26 events (z=2.55), reflecting virtually continuous activity across consecutive 24-hour periods. File access volume on 2023-10-07 was significantly elevated (19 files, z=3.33), specifically involving confidential file access (4 files, z=2.87). The employee repeatedly mounted at least eight distinct USB devices across October 6 and October 7 (E8, E14, E17, E21, E32, E44, E45, E51)."
124.0,2026-11-08T00:03:04.475615,28/28 cited events verified,"The employee may be supporting a time-sensitive cross-departmental audit, migration, or corporate restructuring requiring off-hours access to HR and Finance documentation. The employee could be conducting hardware testing, data backups, or OS provisioning tasks that require cycling through multiple USB storage media. The external emails may have been authorized transmissions of project files to approved external partners or vendors.",gemini-3.8-flash,"Review DLP and email gateway logs for messages E42 and E49 to identify recipients and inspect attachments. Check endpoint logs to determine whether confidential

In [0]:
r = rows[0]
print(f"""
================ DATAGHOST INCIDENT REPORT ================
Employee:  {r['employee_id']}      Severity: {r['severity']}
Risk score: {r['risk_score']:.3f}      Evidence check: {r['grounding']}

SUMMARY
{r['summary']}

WHAT HAPPENED
{r['what_happened']}

WHY IT'S UNUSUAL
{r['why_unusual']}

POSSIBLE INNOCENT EXPLANATIONS
{r['innocent_explanations']}

RECOMMENDED NEXT STEPS
{r['next_steps']}
============================================================
""")


================ DATAGHOST INCIDENT REPORT ================
Employee:  137.0      Severity: High
Risk score: 0.867      Evidence check: 39/39 cited events verified

SUMMARY
Employee 137 exhibited a significant spike in risk score (0.8667) on 2023-10-07, driven by high volumes of after-hours work and access to confidential files. Activity logs indicate continuous round-the-clock operations spanning October 6 and October 7, accompanied by connections to multiple distinct USB devices and regular external emailing. While these patterns warrant review for potential policy violations or data exposure, they may also stem from intensive engineering testing or urgent deployment demands.

WHAT HAPPENED
During the early morning hours of 2023-10-06, the employee accessed files (E2, E4, E5), sent an external email (E1, E7), and connected USB device USB-8314 (E8).
Throughout the business day on 2023-10-06, the employee sent several external emails (E12, E13, E20) and connected additional USB device

In [0]:
cols = events.columns
sample = events.limit(5).toPandas().astype(str).to_dict(orient="records")
victim = str(events.select(EMP).distinct().limit(50).toPandas().sample(1).iloc[0][EMP])

attack_prompt = f"""Invent a realistic but SYNTHETIC insider data-theft scenario for a security demo.
Create 25-40 log events for employee {victim} over 2 days that try to look normal but steal confidential data.
Return a JSON list of objects using EXACTLY these keys: {cols}
Match the value formats in these sample rows: {json.dumps(sample)}"""

import time

attack = None
for attempt in range(6):   # 503s/429s are transient — retry with backoff
    try:
        resp = client.models.generate_content(
            model=GEMINI_MODEL, contents=attack_prompt,
            config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0.9))
        attack = json.loads(resp.text)
        break
    except Exception as e:
        wait = 20 * (attempt + 1)   # 20..120s — long enough for per-minute quota resets
        print(f"  busy, retrying in {wait}s ({str(e)[:60]})")
        time.sleep(wait)
if attack is None:
    raise RuntimeError("Gemini unavailable after retries; try again later.")

attack_df = spark.createDataFrame([{c: str(e.get(c, "")) for c in cols} for e in attack])
attack_df = attack_df.select([F.col(c).cast(events.schema[c].dataType) for c in cols])
display(attack_df)

attack_df.write.mode("append").saveAsTable(f"{DB}.bronze_events")
print(f"👻 Red Ghost injected {attack_df.count()} events for {victim}. "
      "Re-run the features → detection → Gemini notebooks and see if DataGhost catches it!")

  busy, retrying in 20s (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': ')
  busy, retrying in 40s (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': ')
  busy, retrying in 60s (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': ')
  busy, retrying in 80s (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': ')
  busy, retrying in 100s (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': ')
  busy, retrying in 120s (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': ')


---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
File <command-7595439636848305>, line 25
     23         time.sleep(wait)
     24 if attack is None:
---> 25     raise RuntimeError("Gemini unavailable after retries; try again later.")
     27 attack_df = spark.createDataFrame([{c: str(e.get(c, "")) for c in cols} for e in attack])
     28 attack_df = attack_df.select([F.col(c).cast(events.schema[c].dataType) for c in cols])

RuntimeError: Gemini unavailable after retries; try again later.